In [4]:
!pip install -q kagglehub

In [5]:
import gc
import math
import os
import random
import time
import warnings
from pathlib import Path

import kagglehub
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tokenizers import Tokenizer

warnings.filterwarnings("ignore")

In [7]:
# ==========================================================
#                 DOWNLOAD RESOURCES
# ==========================================================

from pathlib import Path
import kagglehub

# -----------------------------
# Download Virgo General Dataset
# -----------------------------
print("Downloading Virgo General Dataset...")

GENERAL_DATASET = Path(
    kagglehub.dataset_download("punitkashyap2007/virgo-general-dataset")
)

# -----------------------------
# Download Virgo Base
# -----------------------------
print("Downloading Virgo Base...")

BASE_MODEL = Path(
    kagglehub.dataset_download("punitkashyap2007/virgo-base-model")
)

print("\nDownloads completed successfully.")

100%|██████████| 1.22G/1.22G [01:23<00:00, 15.7MB/s]

Extracting files...



Downloads completed successfully.


In [8]:
# ==========================================================
#                LOCATE REQUIRED FILES
# ==========================================================

# -----------------------------
# General Dataset
# -----------------------------
train_bin = GENERAL_DATASET / "chat_train.bin"
val_bin = GENERAL_DATASET / "chat_val.bin"
tokenizer_path = GENERAL_DATASET / "virgo_tokenizer.json"

# -----------------------------
# Locate Virgo Base Checkpoint
# -----------------------------
checkpoint = None

for file in BASE_MODEL.rglob("*"):

    if file.suffix in [".pt", ".pth", ".ckpt", ".bin"]:
        checkpoint = file
        break

# -----------------------------
# Verify Files
# -----------------------------
assert train_bin.exists(), "chat_train.bin not found."
assert val_bin.exists(), "chat_val.bin not found."
assert tokenizer_path.exists(), "virgo_tokenizer.json not found."
assert checkpoint is not None, "Virgo Base checkpoint not found."

# -----------------------------
# Display Paths
# -----------------------------
print("=" * 70)
print("Resources")
print("=" * 70)

print(f"Train Bin      : {train_bin}")
print(f"Validation Bin : {val_bin}")
print(f"Tokenizer      : {tokenizer_path}")
print(f"Checkpoint     : {checkpoint}")

print("=" * 70)

Resources
Train Bin      : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-general-dataset/versions/1/chat_train.bin
Validation Bin : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-general-dataset/versions/1/chat_val.bin
Tokenizer      : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-general-dataset/versions/1/virgo_tokenizer.json
Checkpoint     : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-base-model/versions/1/virgo_base_final.pt


In [25]:
# ==========================================================
#                    TRAINING CONFIGURATION
# ==========================================================

class CFG:

    # Model
    vocab_size = 45000
    max_seq_length = 1024

    d_model = 768
    num_heads = 12
    num_layers = 12
    d_ff = 3072
    dropout = 0.10

    # Dataset
    train_bin = train_bin
    val_bin = val_bin
    tokenizer_path = tokenizer_path

    # Pretrained Model
    checkpoint = checkpoint

    # Training
    epochs = 3

    micro_batch_size = 8
    gradient_accumulation_steps = 16

    learning_rate = 5e-5
    min_learning_rate = 5e-6

    weight_decay = 0.01
    warmup_ratio = 0.05

    grad_clip = 1.0

    # Evaluation
    eval_interval = 500

    # Checkpointing
    save_interval = 500

    output_dir = "./virgo_chat"

    save_best = True
    resume_training = True

    last_checkpoint = "virgo_chat_last.pt"
    best_checkpoint = "virgo_chat_best.pt"

    # Reproducibility
    seed = 42


os.makedirs(CFG.output_dir, exist_ok=True)

print("=" * 70)
print("Virgo Chat Configuration")
print("=" * 70)

for key, value in CFG.__dict__.items():
    if not key.startswith("__"):
        print(f"{key:<32}: {value}")

print("=" * 70)

Virgo Chat Configuration
vocab_size                      : 45000
max_seq_length                  : 1024
d_model                         : 768
num_heads                       : 12
num_layers                      : 12
d_ff                            : 3072
dropout                         : 0.1
train_bin                       : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-general-dataset/versions/1/chat_train.bin
val_bin                         : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-general-dataset/versions/1/chat_val.bin
tokenizer_path                  : /root/.cache/kagglehub/datasets/punitkashyap2007/virgo-general-dataset/versions/1/virgo_tokenizer.json


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [26]:
# ==========================================================
#                 SEED & DEVICE SETUP
# ==========================================================

# Set random seed for reproducibility
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


# Initialize seed
set_seed(CFG.seed)


# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Mixed Precision
if torch.cuda.is_available():

    if torch.cuda.is_bf16_supported():
        amp_dtype = torch.bfloat16
    else:
        amp_dtype = torch.float16

else:
    amp_dtype = torch.float32


# Display environment information
print("=" * 70)
print("Environment")
print("=" * 70)

print(f"Device        : {device}")

if device.type == "cuda":

    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version  : {torch.version.cuda}")

    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory    : {total_memory:.2f} GB")

print(f"AMP Data Type : {amp_dtype}")
print(f"Seed          : {CFG.seed}")

print("=" * 70)

Environment
Device        : cuda
GPU           : NVIDIA L4
CUDA Version  : 12.8
GPU Memory    : 22.03 GB
AMP Data Type : torch.bfloat16
Seed          : 42


In [27]:
# ==========================================================
#                    LOAD TOKENIZER
# ==========================================================

# Load tokenizer
tokenizer = Tokenizer.from_file(str(CFG.tokenizer_path))

# Vocabulary information
vocab_size = tokenizer.get_vocab_size()

# Special token IDs
special_tokens = {
    "PAD": tokenizer.token_to_id("<pad>"),
    "UNK": tokenizer.token_to_id("<unk>"),
    "BOS": tokenizer.token_to_id("<bos>"),
    "EOS": tokenizer.token_to_id("<eos>"),
    "NEWLINE": tokenizer.token_to_id("<newline>"),
    "TAB": tokenizer.token_to_id("<tab>")
}

# Display tokenizer information
print("=" * 70)
print("Tokenizer Information")
print("=" * 70)

print(f"Vocabulary Size : {vocab_size:,}")

print("\nSpecial Tokens")
print("-" * 70)

for name, idx in special_tokens.items():
    print(f"{name:<10}: {idx}")

print("=" * 70)

Tokenizer Information
Vocabulary Size : 45,000

Special Tokens
----------------------------------------------------------------------
PAD       : 0
UNK       : 1
BOS       : 2
EOS       : 3
NEWLINE   : 4
TAB       : 5


In [61]:
class BinaryDataset(Dataset):

    def __init__(self, bin_file, seq_len):

        self.seq_len = seq_len

        self.data = np.memmap(
            bin_file,
            dtype=np.uint16,
            mode="r"
        )

        # Number of COMPLETE sequences only
        self.num_sequences = (len(self.data) - seq_len - 1) // seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):

        start = idx * self.seq_len
        end = start + self.seq_len

        x = torch.tensor(
            self.data[start:end],
            dtype=torch.long
        )

        y = torch.tensor(
            self.data[start + 1:end + 1],
            dtype=torch.long
        )

        return x, y

In [29]:
# ==========================================================
#                   CREATE DATASETS
# ==========================================================

# Training dataset
train_dataset = BinaryDataset(
    bin_file=CFG.train_bin,
    seq_len=CFG.max_seq_length
)

# Validation dataset
val_dataset = BinaryDataset(
    bin_file=CFG.val_bin,
    seq_len=CFG.max_seq_length
)

# Display dataset statistics
print("=" * 70)
print("Dataset Information")
print("=" * 70)

print(f"Training Sequences   : {len(train_dataset):,}")
print(f"Validation Sequences : {len(val_dataset):,}")

print(f"\nTraining Tokens      : {len(train_dataset.data):,}")
print(f"Validation Tokens    : {len(val_dataset.data):,}")

print("=" * 70)

Dataset Information
Training Sequences   : 574,457
Validation Sequences : 2,886

Training Tokens      : 588,244,101
Validation Tokens    : 2,956,001


In [90]:
# ==========================================================
#                     DATA LOADERS
# ==========================================================

# Number of workers
num_workers = min(4, os.cpu_count())

# Training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.micro_batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.micro_batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    drop_last=False
)

# Display DataLoader information
print("=" * 70)
print("DataLoader Information")
print("=" * 70)

print(f"Train Batches      : {len(train_loader):,}")
print(f"Validation Batches : {len(val_loader):,}")

print(f"Micro Batch Size   : {CFG.micro_batch_size}")
print(f"Gradient Accum     : {CFG.gradient_accumulation_steps}")
print(f"Effective Batch    : {CFG.micro_batch_size * CFG.gradient_accumulation_steps}")

print(f"Workers            : {num_workers}")

print("=" * 70)

DataLoader Information
Train Batches      : 71,807
Validation Batches : 361
Micro Batch Size   : 8
Gradient Accum     : 16
Effective Batch    : 128
Workers            : 4


In [91]:
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super(RotaryEmbedding, self).__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack((Q_even * cos - Q_odd * sin, Q_even * sin + Q_odd * cos), dim=-1).flatten(-2)
        K = torch.stack((K_even * cos - K_odd * sin, K_even * sin + K_odd * cos), dim=-1).flatten(-2)

        return Q, K

In [92]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [93]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [94]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [95]:
class VirgoModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoModel, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [96]:
# ==========================================================
#                  INITIALIZE MODEL
# ==========================================================

# Build Virgo model
model = VirgoModel(
    vocab_size=CFG.vocab_size,
    d_model=CFG.d_model,
    num_heads=CFG.num_heads,
    num_layers=CFG.num_layers,
    d_ff=CFG.d_ff,
    max_seq_length=CFG.max_seq_length,
    dropout=CFG.dropout
)

# Move model to device
model = model.to(device)

# Display model information
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 70)
print("Model Information")
print("=" * 70)

print(f"Model              : Virgo")
print(f"Total Parameters   : {total_params:,}")
print(f"Trainable Params   : {trainable_params:,}")

print("=" * 70)

Model Information
Model              : Virgo
Total Parameters   : 119,616,000
Trainable Params   : 119,616,000


In [97]:
# ==========================================================
#               LOAD PRETRAINED VIRGO BASE
# ==========================================================

# Checkpoint file path
checkpoint_path = CFG.checkpoint

# If CFG.checkpoint is already a loaded dictionary,
# use it directly. Otherwise load it from disk.
if isinstance(checkpoint_path, dict):
    checkpoint = checkpoint_path
else:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

# Load pretrained weights
model.load_state_dict(checkpoint["model_state_dict"])

# Move model to device
model = model.to(device)

print("=" * 70)
print("Virgo Base Loaded Successfully")
print("=" * 70)

print(f"Epoch        : {checkpoint['epoch']}")
print(f"Step         : {checkpoint['step']}")
print(f"Tokens Seen  : {checkpoint['tokens_seen']:,}")
print(f"Best ValLoss : {checkpoint['best_val_loss']:.4f}")
print(f"Device       : {device}")

print("=" * 70)

Virgo Base Loaded Successfully
Epoch        : 1
Step         : 14038
Tokens Seen  : 1,839,988,736
Best ValLoss : 3.0226
Device       : cuda


In [98]:
# ==========================================================
#          OPTIMIZER, SCHEDULER & MIXED PRECISION
# ==========================================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=CFG.weight_decay
)

# Total optimizer updates
total_updates = (
    len(train_loader) * CFG.epochs
) // CFG.gradient_accumulation_steps

# Learning rate scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_updates,
    eta_min=CFG.min_learning_rate
)

# Automatic Mixed Precision
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(amp_dtype == torch.float16)
)

print("=" * 70)
print("Training Components")
print("=" * 70)

print(f"Optimizer           : AdamW")
print(f"Scheduler           : CosineAnnealingLR")
print(f"Learning Rate       : {CFG.learning_rate:.2e}")
print(f"Minimum LR          : {CFG.min_learning_rate:.2e}")
print(f"Total Updates       : {total_updates:,}")
print(f"AMP Enabled         : {amp_dtype != torch.float32}")
print(f"AMP Data Type       : {amp_dtype}")

print("=" * 70)

Training Components
Optimizer           : AdamW
Scheduler           : CosineAnnealingLR
Learning Rate       : 5.00e-05
Minimum LR          : 5.00e-06
Total Updates       : 13,463
AMP Enabled         : True
AMP Data Type       : torch.bfloat16


In [99]:
# ==========================================================
#         LOSS FUNCTION & TRAINING VARIABLES
# ==========================================================

# Cross Entropy Loss
criterion = nn.CrossEntropyLoss()

# Fine-tuning starts from scratch
start_epoch = 0
global_step = 0

best_val_loss = float("inf")

tokens_processed = 0

print("=" * 70)
print("Fine-Tuning Configuration")
print("=" * 70)

print(f"Start Epoch      : {start_epoch}")
print(f"Global Step      : {global_step}")
print(f"Best Val Loss    : {best_val_loss}")
print(f"Tokens Processed : {tokens_processed:,}")

print("=" * 70)

Fine-Tuning Configuration
Start Epoch      : 0
Global Step      : 0
Best Val Loss    : inf
Tokens Processed : 0


In [100]:
# ==========================================================
#                  TRAIN ONE EPOCH
# ==========================================================

def train_one_epoch(epoch):

    model.train()

    running_loss = 0.0

    running_correct = 0
    running_tokens = 0

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{CFG.epochs}",
        leave=False
    )

    global global_step
    global tokens_processed

    for step, (x, y) in enumerate(progress_bar):

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=(amp_dtype != torch.float32)
        ):

            logits = model(x)

            loss = criterion(
                logits.view(-1, CFG.vocab_size),
                y.view(-1)
            )

            loss = loss / CFG.gradient_accumulation_steps

        scaler.scale(loss).backward()

        predictions = logits.argmax(dim=-1)

        # Current batch accuracy
        current_correct = (predictions == y).sum().item()
        current_tokens = y.numel()

        current_acc = 100.0 * current_correct / current_tokens

        # Running statistics
        running_correct += current_correct
        running_tokens += current_tokens

        running_loss += loss.item() * CFG.gradient_accumulation_steps

        running_acc = 100.0 * running_correct / running_tokens

        # Actual processed tokens
        tokens_processed += current_tokens

        if (step + 1) % CFG.gradient_accumulation_steps == 0:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CFG.grad_clip
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            scheduler.step()

            global_step += 1

        current_loss = running_loss / (step + 1)

        progress_bar.set_postfix(
            Loss=f"{current_loss:.4f}",
            CurAcc=f"{current_acc:.2f}%",
            RunAcc=f"{running_acc:.2f}%",
            Tokens=f"{tokens_processed/1e6:.2f}M",
            LR=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_acc

    return epoch_loss, epoch_acc

In [101]:
# ==========================================================
#                   VALIDATION FUNCTION
# ==========================================================

@torch.no_grad()
def validate():

    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_tokens = 0

    progress_bar = tqdm(
        val_loader,
        desc="Validation",
        leave=False
    )

    for x, y in progress_bar:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=(amp_dtype != torch.float32)
        ):

            logits = model(x)

            loss = criterion(
                logits.view(-1, CFG.vocab_size),
                y.view(-1)
            )

        predictions = logits.argmax(dim=-1)

        running_correct += (predictions == y).sum().item()
        running_tokens += y.numel()

        running_loss += loss.item()

        progress_bar.set_postfix(
            Loss=f"{running_loss / (progress_bar.n + 1):.4f}",
            Acc=f"{100 * running_correct / running_tokens:.2f}%"
        )

    val_loss = running_loss / len(val_loader)
    val_acc = 100 * running_correct / running_tokens

    return val_loss, val_acc

In [102]:
# ==========================================================
#                  SAVE CHECKPOINT
# ==========================================================

BEST_MODEL_PATH = os.path.join(CFG.output_dir, "virgo_chat_best.pt")
LAST_MODEL_PATH = os.path.join(CFG.output_dir, "virgo_chat_last.pt")


def save_checkpoint(epoch, val_loss):

    global best_val_loss

    epoch_model_path = os.path.join(
        CFG.output_dir,
        f"virgo_chat_epoch_{epoch}.pt"
    )

    checkpoint = {
        "epoch": epoch,
        "step": global_step,
        "tokens_seen": tokens_processed,
        "best_val_loss": best_val_loss,

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),

        "config": {
            "vocab_size": CFG.vocab_size,
            "d_model": CFG.d_model,
            "num_heads": CFG.num_heads,
            "num_layers": CFG.num_layers,
            "d_ff": CFG.d_ff,
            "max_seq_length": CFG.max_seq_length,
            "dropout": CFG.dropout,
        }
    }

    # Save latest checkpoint
    torch.save(checkpoint, LAST_MODEL_PATH)

    # Save checkpoint for every epoch
    torch.save(checkpoint, epoch_model_path)

    print(f"Epoch {epoch} checkpoint saved.")

    # Save best checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        checkpoint["best_val_loss"] = best_val_loss

        torch.save(checkpoint, BEST_MODEL_PATH)

        print(f"Best model saved. Validation Loss: {val_loss:.4f}")

In [103]:
# ==========================================================
#                    TRAINING LOOP
# ==========================================================

print("=" * 70)
print("Starting Virgo Chat Fine-Tuning")
print("=" * 70)

training_start_time = time.time()

for epoch in range(start_epoch, CFG.epochs):

    epoch_start_time = time.time()

    # --------------------------
    # Train
    # --------------------------
    train_loss, train_acc = train_one_epoch(epoch)

    # --------------------------
    # Validate
    # --------------------------
    val_loss, val_acc = validate()

    # --------------------------
    # Save Checkpoint
    # --------------------------
    save_checkpoint(epoch + 1, val_loss)

    epoch_time = time.time() - epoch_start_time

    print("\n" + "=" * 70)
    print(f"Epoch {epoch + 1}/{CFG.epochs}")
    print("=" * 70)

    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.2f}%")

    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Acc  : {val_acc:.2f}%")

    print(f"Learning Rate   : {scheduler.get_last_lr()[0]:.2e}")
    print(f"Global Step     : {global_step:,}")
    print(f"Tokens Seen     : {tokens_processed:,}")

    print(f"Epoch Time      : {epoch_time / 60:.2f} min")

    print("=" * 70)

training_time = time.time() - training_start_time

print("\n")
print("=" * 70)
print("Training Completed Successfully")
print("=" * 70)

print(f"Training Time        : {training_time / 3600:.2f} hours")
print(f"Best Validation Loss : {best_val_loss:.4f}")
print(f"Global Steps         : {global_step:,}")
print(f"Tokens Processed     : {tokens_processed:,}")

print("\nSaved Models")

print(f"Latest : {LAST_MODEL_PATH}")
print(f"Best   : {BEST_MODEL_PATH}")
print(f"Epochs : {CFG.output_dir}")

print("=" * 70)

Starting Virgo Chat Fine-Tuning


Epoch 1/3:   0%|          | 0/71807 [00:00<?, ?it/s]

Validation:   0%|          | 0/361 [00:00<?, ?it/s]

Epoch 1 checkpoint saved.
Best model saved. Validation Loss: 2.1344

Epoch 1/3
Train Loss      : 2.3286
Train Accuracy  : 52.93%
Validation Loss : 2.1344
Validation Acc  : 55.75%
Learning Rate   : 3.88e-05
Global Step     : 4,487
Tokens Seen     : 588,242,944
Epoch Time      : 319.15 min


Epoch 2/3:   0%|          | 0/71807 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [104]:
# ==========================================================
#                    GENERATE RESPONSE
# ==========================================================

@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=256,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1
):

    model.eval()

    text = (
        "<bos>\n"
        "User:\n"
        f"{prompt.strip()}\n\n"
        "Virgo:\n"
    )

    input_ids = tokenizer.encode(text).ids
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)

    eos_id = tokenizer.token_to_id("<eos>")

    for _ in range(max_new_tokens):

        # Keep only last context window
        input_ids = input_ids[:, -CFG.max_seq_length:]

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=(amp_dtype != torch.float32)
        ):
            logits = model(input_ids)

        logits = logits[:, -1, :]

        # Repetition penalty
        if repetition_penalty != 1.0:

            for token_id in set(input_ids[0].tolist()):
                logits[:, token_id] /= repetition_penalty

        # Temperature
        logits = logits / temperature

        # Top-K
        if top_k > 0:

            values, _ = torch.topk(logits, top_k)

            logits[logits < values[:, [-1]]] = -float("inf")

        # Top-P
        if top_p < 1.0:

            sorted_logits, sorted_indices = torch.sort(
                logits,
                descending=True
            )

            probs = torch.softmax(sorted_logits, dim=-1)

            cumulative_probs = torch.cumsum(probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False

            indices_to_remove = sorted_indices[
                sorted_indices_to_remove
            ]

            logits[:, indices_to_remove] = -float("inf")

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, 1)

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

        if next_token.item() == eos_id:
            break

    output = tokenizer.decode(
        input_ids[0].tolist(),
        skip_special_tokens=False
    )

    if "Virgo:" in output:
        output = output.split("Virgo:", 1)[1]

    if "<eos>" in output:
        output = output.split("<eos>")[0]

    return output.strip()

In [105]:
while True:

    prompt = input("\nUser: ")

    if prompt.lower() in ["exit", "quit", "bye"]:
        break

    response = generate(
        prompt,
        max_new_tokens=256,
        temperature=0.7,
        top_k=40,
        top_p=0.9
    )

    print("\nVirgo:", response)


User:  hi



Virgo: <unk>Sure, here are some additional details on how to create a virtual assistant in Node.js that can perform CRUD operations on a website and handle user input: 1. Create an API endpoint for your website: To start the application, you need to create an API endpoint for your website. Here's an example of how you could do this: ```javascript const express = require('express'); const app = express(); const app = express(); const router = http.Router(); router.get('/', (req, res) => <unk> res.sendRequest('Request request: ' + req.body.statusCode()); }); ``` In this code, we first import the necessary dependencies by adding `express` and `req.body.statusCode()`. Then, we define a route `/` using the `req` module. Inside the route handler function, we check if the request is successful and return a response with the message "Request request:". If it is not successful, we render a form element and add the response body to it. 2. Implement a JavaScript client to handle



User:  introduce yourself ?



Virgo: <unk>Introducing yourself, a world-class superhero who has become one of the most respected and feared superheroes in the universe. As an accomplished superhero, you will always be a source of inspiration and guidance for aspiring superheroes. With our charismatic and innovative minds, we can help you navigate the challenges of becoming a superhero and find your true calling. Whether you're a seasoned professional or just looking for inspiration, our superheroes are not only a fun and rewarding way to stay alive but also a beacon of hope for future generations.<unk><unk>



User:  what is physics



Virgo: <unk>Science is a branch of science that deals with the study of matter and energy, which are both fundamental concepts in the field of physics. It is based on the laws of physics and is based on the principles of physics, such as gravity, force, and conservation of energy. In other words, the key to understanding the fundamental nature of matter is to understand how it behaves in different ways, which can be applied in various applications. In physics, physics refers to the study of matter at the atomic level, which involves the study of subatomic particles and their interactions with matter. Understanding this aspect is crucial for understanding the behavior of matter and energy in various fields such as physics, physics, and biology.<unk><unk>



User:  what is  2 + 2 = ?



Virgo: <unk>The question asks for the area of a rectangle with a length of 3 units and a width of 4 units. The area of a rectangle is calculated by multiplying its length (length) by the width (width). So, the area of the rectangle is 2 * 3 = 6 square units. Since there are 12 units in a cubic foot, the area of the rectangle is 6 * 12 = 72 square units. #### 72 The answer is: 72<unk><unk>



User:  exit


In [ ]:
# ==========================================================
#                    GENERATE RESPONSE
# ==========================================================

import time
from IPython.display import display, Markdown


def typewriter(text, delay=0.015):
    """
    Smooth typewriter effect for Jupyter/Kaggle.
    """
    handle = display(Markdown(""), display_id=True)

    current = ""

    for ch in text:
        current += ch
        handle.update(Markdown(f"```text\n{current}\n```"))
        time.sleep(delay)


@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=256,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
    stream=True,
    typing_delay=0.015
):

    model.eval()

    text = (
        "<bos>\n"
        "User:\n"
        f"{prompt.strip()}\n\n"
        "Virgo:\n"
    )

    input_ids = tokenizer.encode(text).ids
    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )

    eos_id = tokenizer.token_to_id("<eos>")

    for _ in range(max_new_tokens):

        # Keep only context window
        input_ids = input_ids[:, -CFG.max_seq_length:]

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=(amp_dtype != torch.float32)
        ):
            logits = model(input_ids)

        logits = logits[:, -1, :]

        # -----------------------------
        # Repetition Penalty
        # -----------------------------
        if repetition_penalty != 1.0:

            for token_id in set(input_ids[0].tolist()):

                if logits[:, token_id] < 0:
                    logits[:, token_id] *= repetition_penalty
                else:
                    logits[:, token_id] /= repetition_penalty

        # -----------------------------
        # Temperature
        # -----------------------------
        logits /= temperature

        # -----------------------------
        # Top-K Sampling
        # -----------------------------
        if top_k > 0:

            values, _ = torch.topk(logits, top_k)

            logits[logits < values[:, [-1]]] = -float("inf")

        # -----------------------------
        # Top-P Sampling
        # -----------------------------
        if top_p < 1.0:

            sorted_logits, sorted_indices = torch.sort(
                logits,
                descending=True
            )

            sorted_probs = torch.softmax(
                sorted_logits,
                dim=-1
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1
            )

            sorted_indices_to_remove = cumulative_probs > top_p

            sorted_indices_to_remove[..., 1:] = (
                sorted_indices_to_remove[..., :-1].clone()
            )

            sorted_indices_to_remove[..., 0] = False

            indices_to_remove = sorted_indices[
                sorted_indices_to_remove
            ]

            logits[:, indices_to_remove] = -float("inf")

        # -----------------------------
        # Sample Next Token
        # -----------------------------
        probs = torch.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

        if next_token.item() == eos_id:
            break

    # ==========================================================
    # Decode
    # ==========================================================

# ==========================================================
# Decode
# ==========================================================

    output = tokenizer.decode(
        input_ids[0].tolist(),
        skip_special_tokens=False
    )

    # Remove chat prefix
    if "Virgo:" in output:
        output = output.split("Virgo:", 1)[1]

    # Stop at EOS
    if "<eos>" in output:
        output = output.split("<eos>", 1)[0]

    # Remove remaining special tokens
    for token in [
        "<bos>",
        "<unk>",
        "<pad>",
        "<eos>"
    ]:
        output = output.replace(token, "")

    # Restore formatting
    output = output.replace("<newline>", "\n")
    output = output.replace("<tab>", "\t")

    # Clean whitespace while preserving newlines
    lines = [line.strip() for line in output.split("\n")]
    output = "\n".join(line for line in lines if line)

    output = output.strip()
    # ==========================================================
    # Display
    # ==========================================================

    if stream:
        typewriter(output, delay=typing_delay)
    else:
        print(output)

    return output


# ==========================================================
#                   CHAT LOOP
# ==========================================================

print("=" * 60)
print("🌌 Virgo Chat")
print("Type 'exit' to quit.")
print("=" * 60)

while True:

    prompt = input("\n🧑 You: ")

    if prompt.lower() in [
        "exit",
        "quit",
        "bye"
    ]:
        print("\n👋 Goodbye!")
        break

    print("\n🤖 Virgo:\n")

    generate(
        prompt,
        max_new_tokens=256,
        temperature=0.7,
        top_k=40,
        top_p=0.90,
        repetition_penalty=1.15,
        stream=True,
        typing_delay=0.01
    )

🌌 Virgo Chat
Type 'exit' to quit.



🧑 You:  Complete the sequence:  2, 4, 6, 8,



🤖 Virgo:



```text
The sequence is as follows: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 47, 49, 50, 51, 52, 53, 59, 60, 61, 67, 68, 69, 70, 71, 73, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 100, 101, 103, 104, 105, 106, 107, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126, 126,
```


🧑 You:  Read the paragraph carefully.  Emma went to the library on Monday and borrowed three books about space. On Tuesday, she finished reading one book. On Wednesday, she returned that book and borrowed another one about dinosaurs. By Friday, she had read all three books she still had.  Question: How many books did Emma have on Friday?


...


🤖 Virgo:



```text
I'm sorry to hear that! Here's a simple way of saying it in a positive manner: "But, what do you want to say?" I can imagine the word "laughing" and the speaker is talking about their own feelings and thoughts. They may feel annoyed or frustrated, but they don't know how to react. They might be surprised by their fear and frustration.
```


🧑 You:  Complete this sentence:  The capital of France is



🤖 Virgo:



```text
The capital of France is the French capital, which stands for "French" and has its own name.
```